# 01 — Data Setup

This notebook walks through every step needed to go from a blank slate to
a clean set of training-ready parquets:

1. Load your `config.yaml` and display a summary.
2. Draw your bounding box on an interactive map and confirm the ASOS stations inside it.
3. Download ASOS observations and resample them to 1-hourly parquets.
4. Download HRRR forecasts for every forecast hour in your config.
5. Quality-check the data: availability heatmap (station × date, % missing).
6. Build the station-cluster lookup parquet that the model needs.

**Run each cell in order.**  Heavy download steps (HRRR, ASOS) can take
minutes to hours depending on your date range and internet connection.
All outputs are written to `./output/` (relative to `app/`) and every
step is resume-friendly — re-running a cell skips already-downloaded files.

In [ ]:
import sys, os

# Make sure the repo root (one level above app/) is on the Python path
# so that both `app.*` and `src.*` modules resolve correctly.
REPO_ROOT = os.path.abspath("../..")
APP_ROOT  = os.path.abspath("..")
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
if APP_ROOT not in sys.path:
    sys.path.insert(0, APP_ROOT)

os.chdir(APP_ROOT)  # resolve relative paths in config.yaml from app/
print("Working directory:", os.getcwd())

In [ ]:
from app.utils.config_loader import load_config

cfg = load_config()   # reads app/config.yaml
cfg.make_output_dirs()

print("=== Configuration ===")
print(f"  Bounding box  : lat=[{cfg.bbox.lat_min}, {cfg.bbox.lat_max}]"
      f"  lon=[{cfg.bbox.lon_min}, {cfg.bbox.lon_max}]")
print(f"  Mesonet source: {cfg.data.mesonet_source}")
print(f"  Forecast hours: {cfg.data.forecast_hours}")
print(f"  Met variables : {cfg.data.metvars}")
print(f"  Model type    : {cfg.model}")
print(f"  Training      : {cfg.training.start_date} → {cfg.training.end_date}")
print(f"  Train/Val/Test: {cfg.training.train_frac:.0%} / "
      f"{cfg.training.val_frac:.0%} / {cfg.training.test_frac:.0%}")
print(f"  Device id     : {cfg.training.device_id}")

## Step 1 — Visualise the bounding box and ASOS stations

In [ ]:
import folium
from app.utils.bbox_filter import bbox_corners
from app.data.fetch_asos import get_asos_stations_in_bbox

# Centre the map on the bbox midpoint
centre_lat = (cfg.bbox.lat_min + cfg.bbox.lat_max) / 2
centre_lon = (cfg.bbox.lon_min + cfg.bbox.lon_max) / 2
m = folium.Map(location=[centre_lat, centre_lon], zoom_start=7, tiles="CartoDB positron")

# Draw the bbox polygon
corners = bbox_corners(cfg.bbox)
folium.Polygon(
    locations=corners,
    color="#e63946",
    weight=2,
    fill=True,
    fill_opacity=0.08,
    tooltip="Bounding box",
).add_to(m)

# Query ASOS station inventory and add markers
print("Fetching ASOS station inventory …")
station_meta = get_asos_stations_in_bbox(
    cfg.bbox.lat_min, cfg.bbox.lat_max, cfg.bbox.lon_min, cfg.bbox.lon_max
)
print(f"  {len(station_meta)} stations found inside the bbox.")

for _, row in station_meta.iterrows():
    folium.CircleMarker(
        location=[row.lat, row.lon],
        radius=5,
        color="#457b9d",
        fill=True,
        fill_opacity=0.8,
        tooltip=f"{row.station} — {row.get('name', '')}",
    ).add_to(m)

display(m)

In [ ]:
# Preview the station metadata table
station_meta.head(10)

## Step 2 — Download ASOS observations

This cell calls the IEM REST API in 30-day chunks.  For a 2-year window with
~60 stations it typically takes 5–15 minutes.  Already-downloaded years are
skipped automatically.

In [ ]:
from app.data.fetch_asos import fetch_asos_for_bbox

if cfg.data.mesonet_source == "asos":
    station_meta = fetch_asos_for_bbox(
        cfg,
        cfg.training.start_date,
        cfg.training.end_date,
        show_progress=True,
    )
    print("\nASOS download complete.")
else:
    # User-supplied local mesonet — just copy the station metadata
    import pandas as pd, shutil
    from pathlib import Path
    assert cfg.data.station_meta_csv is not None, (
        "Set data.station_meta_csv in config.yaml when using mesonet_source='local'"
    )
    station_meta = pd.read_csv(cfg.data.station_meta_csv)
    out_path = Path(cfg.output.models_dir) / "lookups" / "station_meta.csv"
    shutil.copy(cfg.data.station_meta_csv, out_path)
    print(f"Copied local station metadata to {out_path}")

## Step 3 — Download HRRR forecasts

> **Warning**: Downloading years of raw HRRR grib2 data is disk- and
> time-intensive.  Each grib2 file is ~50–200 MB; only the variables the
> model needs are extracted before writing the parquet.
>
> If you already have pre-cleaned HRRR parquets in the expected layout,
> set `hrrr_raw_dir` in `config.yaml` to point to them and skip this cell.

In [ ]:
from app.data.fetch_hrrr import fetch_and_stage

fetch_and_stage(
    cfg,
    cfg.training.start_date,
    cfg.training.end_date,
    station_lookup=station_meta,
    show_progress=True,
)
print("HRRR staging complete.")

## Step 4 — Data availability quality check

This heatmap shows the fraction of hourly observations that are non-missing
for each station and calendar day.  Aim for ≥ 80 % coverage before training.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

mesonet_dir = Path(cfg.data.parquets_dir) / "mesonet"
all_obs = []
for pq in sorted(mesonet_dir.glob("mesonet_1H_obs_*.parquet")):
    df = pd.read_parquet(pq)
    if df.index.nlevels > 1:
        df = df.reset_index()
    all_obs.append(df)

if not all_obs:
    print("No mesonet parquets found — run the ASOS download step first.")
else:
    obs = pd.concat(all_obs, ignore_index=True)
    obs["time_1H"] = pd.to_datetime(obs["time_1H"], utc=True, errors="coerce")
    obs["date"] = obs["time_1H"].dt.date

    # Fraction of non-missing tair observations per (station, date)
    avail = (
        obs.groupby(["station", "date"])["tair"]
        .apply(lambda s: (s != -999).mean())
        .unstack("station")
    )
    avail.index = pd.to_datetime(avail.index)

    fig, ax = plt.subplots(figsize=(max(12, len(avail.columns) * 0.3), 6))
    im = ax.imshow(
        avail.T.values,
        aspect="auto",
        vmin=0, vmax=1,
        cmap="RdYlGn",
        extent=[
            mdates.date2num(avail.index[0].to_pydatetime()),
            mdates.date2num(avail.index[-1].to_pydatetime()),
            -0.5, len(avail.columns) - 0.5,
        ],
    )
    ax.xaxis_date()
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    ax.set_yticks(range(len(avail.columns)))
    ax.set_yticklabels(avail.columns, fontsize=7)
    plt.colorbar(im, ax=ax, label="Fraction available (tair)")
    ax.set_title("Data availability — tair")
    plt.tight_layout()
    plt.savefig(Path(cfg.output.results_dir) / "data_availability.png", dpi=150)
    plt.show()
    print(f"\nMedian availability: {avail.median().median():.1%}")

## Step 5 — Build station clusters

The model takes **four stations** as input: the target station plus its three
nearest neighbours.  This step computes and caches that lookup table.

In [ ]:
from app.data.build_station_clusters import build_clusters

lookup_df = build_clusters(cfg, station_meta=station_meta, show_progress=True)
print(f"\nLookup table has {len(lookup_df)} entries.")
lookup_df.head()

## All done!

Your output directory should now contain:

```
output/
├── parquets/
│   ├── mesonet/mesonet_1H_obs_{YYYY}.parquet
│   └── hrrr_data/fh{N}/HRRR_{YYYY}_{MM}_...parquet
├── models/lookups/
│   ├── station_meta.csv
│   └── triangulate_stations.parquet
└── results/data_availability.png
```

Open **`02_train.ipynb`** to train the model.